# 07 · SAR como herramienta explicativa: secciones que rinden por encima o debajo de su perfil socioeconómico

**Objetivo.** Ajustar un modelo espacial (SAR, *spatial autoregressive*) del voto al
oficialismo (Morena) en 2024 sobre el perfil socioeconómico de cada sección, y usar sus
**residuos** como herramienta explicativa: identificar secciones que votan *más*
oficialista (o *menos*) de lo que su demografía y su entorno territorial predicen.

**Advertencia metodológica (honestidad ante todo).** Este modelo es **explicativo, no
predictivo**: es transversal (solo 2024), sin dimensión temporal. El backtest de la Fase 3
mostró que los modelos territoriales no baten al swing uniforme fuera de era, así que
este notebook NO se usa para pronosticar 2027. Su valor es **diagnóstico**: ¿dónde
hay voto oficialista "desproporcionado" a su base socioeconómica, y dónde resistencia
"inusual"? Esas secciones sugieren factores locales (candidaturas, clientelismo,
organización territorial, identidad regional) que ningún censo captura.

**Especificación.**
- Variable dependiente: `pct_morena_2024` (% del voto a Morena en la sección).
- Covariables (Censo 2020 a sección): educación, vivienda, bienes, salud, edad, ruralidad.
- Pesos espaciales: contigüidad Queen, estandarizada por filas.
- Estimador: **GM_Lag** (Kelejian-Prucha, GMM) — con N≈53,000 la máxima verosimilitud
  exacta es inviable; GM_Lag es consistente y escala a matrices dispersas.

In [1]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from libpysal.weights import Queen
from spreg import GM_Lag, OLS
from esda.moran import Moran, Moran_Local

from agente.panel import cargar_panel
from agente.cartografia import cargar_secciones

RAIZ = os.path.abspath("..")
FIG = os.path.join(RAIZ, "figuras")
os.makedirs(FIG, exist_ok=True)
print("librerías listas")

librerías listas


## 1. Datos: panel + cartografía de secciones

In [2]:
panel = cargar_panel()
secciones = cargar_secciones()
g = secciones[["id", "geometry"]].merge(panel, on="id", how="inner")
COVS = ["pct_sin_basica", "pct_analfabeta", "pct_piso_tierra", "pct_sin_drenaje",
        "pct_sin_bienes", "pct_auto", "pct_internet", "pct_sin_salud",
        "pct_60ymas", "pct_18a24", "pct_rural"]
d = g.dropna(subset=["pct_morena_2024"] + COVS).reset_index(drop=True)
print(f"Secciones con voto 2024 y censo completo: {len(d):,} de {len(panel):,}")
print(f"(las ~18,500 restantes no tienen censo desagregado — vacíos documentados del Censo 2020)")

Secciones con voto 2024 y censo completo: 52,976 de 71,541
(las ~18,500 restantes no tienen censo desagregado — vacíos documentados del Censo 2020)


## 2. Autocorrelación espacial del voto

Antes de modelar: ¿qué tan agrupado está el voto a Morena en el espacio?

In [3]:
w = Queen.from_dataframe(d[["geometry"]], use_index=False, silence_warnings=True)
w.transform = "r"
moran_y = Moran(d["pct_morena_2024"].values, w)
print(f"Moran's I de pct_morena_2024 (Queen, fila-estándar): {moran_y.I:.3f}")
print(f"p-valor (permutaciones): {moran_y.p_sim:.4f}  → autocorrelación fuertísima")
print(f"Vecinos promedio por sección: {w.mean_neighbors:.1f}")

Moran's I de pct_morena_2024 (Queen, fila-estándar): 0.843
p-valor (permutaciones): 0.0010  → autocorrelación fuertísima
Vecinos promedio por sección: 5.3


In [4]:
# mapa del voto a Morena 2024 (contexto)
fig, ax = plt.subplots(figsize=(9, 7))
dm = d.copy()
dm["geometry"] = dm.geometry.simplify(0.001)
dm.plot(column="pct_morena_2024", ax=ax, cmap="RdBu_r", legend=True,
        linewidth=0, vmin=0, vmax=100,
        legend_kwds={"label": "% voto Morena 2024", "shrink": 0.7})
ax.set_title("Voto a Morena por sección, elección 2024", fontsize=12)
ax.set_axis_off()
fig.savefig(os.path.join(FIG, "sar07_voto_morena.png"), dpi=110, bbox_inches="tight")
plt.show()
plt.close(fig)

## 3. OLS con diagnósticos espaciales

Primero el modelo ingenuo (sin estructura espacial) para tener línea base y confirmar
la necesidad del término espacial con las pruebas de Lagrange.

In [5]:
y = d[["pct_morena_2024"]].values
X = d[COVS].values
ols = OLS(y, X, w=w, spat_diag=True, moran=True, name_y="pct_morena_2024", name_w="queen")
print(f"OLS: R² = {ols.r2:.3f}; AIC = {ols.aic:.0f}")
print(f"Moran's I de los residuos OLS: {ols.moran_res[0]:.3f} (z={ols.moran_res[1]:.1f})")
print(f"LM-lag: {ols.lm_lag[0]:.0f} (p={ols.lm_lag[1]:.4f})   LM-error: {ols.lm_error[0]:.0f} (p={ols.lm_error[1]:.4f})")
print("→ Ambas pruebas rechazan OLS: hay dependencia espacial (lag y error). Procedemos con SAR.")

OLS: R² = 0.257; AIC = 417579
Moran's I de los residuos OLS: 0.824 (z=286.6)
LM-lag: 67653 (p=0.0000)   LM-error: 82043 (p=0.0000)
→ Ambas pruebas rechazan OLS: hay dependencia espacial (lag y error). Procedemos con SAR.


## 4. Modelo SAR (GM_Lag)

`y = ρ·Wy + Xβ + ε` — el voto de una sección depende del voto de sus vecinas (ρ)
y de su perfil socioeconómico (β), más un choque local (ε). El residuo ε es la
**desviación inexplicada**: lo que ni el censo ni el entorno capturan.

In [6]:
sar = GM_Lag(y, X, w=w, w_lags=2, name_y="pct_morena_2024", name_w="queen")
betas = {c: round(b, 3) for c, b in zip(COVS, sar.betas[:-2, 0])}
print(f"ρ (autoregresivo espacial) = {sar.rho[0]:.3f}")
print(f"pseudo-R² = {sar.pr2:.3f}   (OLS era {ols.r2:.3f})")
print("\nBetas (efecto directo aproximado):")
for k, v in betas.items():
    print(f"  {k:20s} {v:+.3f}")

GM_Lag
ρ (autoregresivo espacial) = 0.270
pseudo-R² = 0.589   (OLS era 0.257)

Betas (efecto directo aproximado):
  pct_sin_basica       +28.864
  pct_analfabeta       +0.162
  pct_piso_tierra      -0.427
  pct_sin_drenaje      +0.223
  pct_sin_bienes       +0.003
  pct_auto             -0.192
  pct_internet         -0.224
  pct_sin_salud        -0.048
  pct_60ymas           -0.037
  pct_18a24            -0.361
  pct_rural            +0.046


Lectura de coeficientes (efecto *ceteris paribus* sobre el % de voto a Morena):

- **`pct_sin_basica`** es el más fuerte: +1 pp de población sin educación básica →
  ≈ +0.29 pp de voto a Morena. La base educativa del oficialismo es estructural.
- **`pct_auto`** y **`pct_internet`** son negativos: prosperidad material se asocia
  con menos voto oficialista.
- **ρ ≈ 0.27**: controlando por socioeconomía, cada sección "hereda" parte del voto
  de sus vecinas — organización territorial, difusión local de opiniones. Es el
  mecanismo que el backtest de 2027 no logró explotar predictivamente, pero que
  aquí confirma que la geografía importa *dentro* de la era política actual.

In [7]:
d["pred"] = sar.predy[:, 0]
d["resid"] = d["pct_morena_2024"] - d["pred"]
print(f"Residuos: media {d['resid'].mean():+.2f}, desv. est. {d['resid'].std():.2f} pts")
print(f"Rango: [{d['resid'].min():.1f}, {d['resid'].max():.1f}] pts")
moran_res = Moran(d["resid"].values, w)
print(f"Moran's I de residuos SAR: {moran_res.I:.3f} (p={moran_res.p_sim:.4f})")
print("→ El residuo aún está espacialmente agrupado: los 'factores locales' también son territoriales.")

Residuos: media -0.00, desv. est. 9.75 pts
Rango: [-52.3, 58.8] pts


Moran's I de residuos SAR: 0.703 (p=0.0010)
→ El residuo aún está espacialmente agrupado: los 'factores locales' también son territoriales.


## 5. Secciones sobre- y sub-performantes

Un residuo muy **positivo** = sección que votó *más* oficialista de lo que su perfil
predice (máquinas territoriales, clientelismo, candidaturas locales fuertes).
Un residuo muy **negativo** = resistencia local inusual (identidad regional,
oposición organizada, descontento hiperlocal).

Clasificamos con umbrales de ±1.5 desviaciones estándar (~±15 pts).

In [8]:
sd = d["resid"].std()
d["categoria_sar"] = np.where(d["resid"] >  1.5*sd, "sobre_oficialista",
                       np.where(d["resid"] < -1.5*sd, "sub_oficialista", "en_lo_esperado"))
print(d["categoria_sar"].value_counts())
cat = d.groupby("categoria_sar").agg(
    n=("id","count"), resid_medio=("resid","mean"), morena_medio=("pct_morena_2024","mean"),
    pred_medio=("pred","mean")).round(1)
cat

categoria_sar
en_lo_esperado       45801
sobre_oficialista     3983
sub_oficialista       3192
Name: count, dtype: int64


,n,resid_medio,morena_medio,pred_medio
categoria_sar,,,,
en_lo_esperado,45801,-0.3,42.4,42.7
sobre_oficialista,3983,19.0,66.1,47.1
sub_oficialista,3192,-19.4,20.7,40.1


In [9]:
# mapa nacional de residuos
fig, ax = plt.subplots(figsize=(9, 7))
dm = d.copy(); dm["geometry"] = dm.geometry.simplify(0.001)
dm.plot(column="resid", ax=ax, cmap="RdBu_r", legend=True, linewidth=0,
        vmin=-25, vmax=25, legend_kwds={"label": "residuo SAR (pts de voto Morena)", "shrink": 0.7})
ax.set_title("Residuos del SAR: voto a Morena por encima (rojo) o debajo (azul)\nde lo que su perfil socioeconómico predice (2024)", fontsize=12)
ax.set_axis_off()
fig.savefig(os.path.join(FIG, "sar07_residuos.png"), dpi=110, bbox_inches="tight")
plt.show()
plt.close(fig)

## 6. Clusters territoriales de residuos (LISA)

¿Dónde se concentran las desviaciones? Usamos LISA sobre los residuos del SAR:
clusters Alto-Alto = zonas sistemáticamente más oficialistas que su perfil;
Bajo-Bajo = resistencia territorial estructural.

In [10]:
lisa_res = Moran_Local(d["resid"].values, w, permutations=199)
sig = lisa_res.p_sim < 0.05
cuadrantes = np.where(~sig, "no_sig",
              np.where((lisa_res.q==1) & (lisa_res.Is>0), "AA_mas_oficialista",
              np.where((lisa_res.q==2) & (lisa_res.Is<0), "BA_resistencia",
              np.where((lisa_res.q==3) & (lisa_res.Is<0), "BB_resistencia",
              "AB_mas_oficialista"))))
d["lisa_resid"] = cuadrantes
print(d.loc[sig, "lisa_resid"].value_counts())

lisa_resid
AB_mas_oficialista    10019
AA_mas_oficialista     8775
BA_resistencia          388
Name: count, dtype: int64


In [11]:
fig, ax = plt.subplots(figsize=(9, 7))
dm = d.copy(); dm["geometry"] = dm.geometry.simplify(0.001)
colores = {"AA_mas_oficialista": "#b2182b", "AB_mas_oficialista": "#ef8a62",
           "BA_resistencia": "#67a9cf", "BB_resistencia": "#2166ac", "no_sig": "#eeeeee"}
for c, col in colores.items():
    gsub = dm[dm["lisa_resid"] == c]
    if len(gsub):
        gsub.plot(ax=ax, color=col, label=c.replace("_", " "), linewidth=0)
ax.legend(loc="lower left", fontsize=8, title="LISA de residuos (p<0.05)")
ax.set_title("Clusters de secciones que se desvían del modelo socioeconómico", fontsize=12)
ax.set_axis_off()
fig.savefig(os.path.join(FIG, "sar07_lisa_residuos.png"), dpi=110, bbox_inches="tight")
plt.show()
plt.close(fig)

## 7. ¿Quiénes son? Las secciones más desviadas del país

In [12]:
from agente.panel import cargar_catalogo
cat = cargar_catalogo()
nombres = cat.drop_duplicates("ent")[["ent", "nom_ent"]].set_index("ent")["nom_ent"].to_dict()
mun_nombres = cat.set_index(["ent", "mun"])["nom_mun"].to_dict()
d["nom_mun"] = [mun_nombres.get((e, m), "") for e, m in zip(d["ent"], d["mun"])]
d["nom_ent"] = d["ent"].map(nombres)

cols = ["id", "nom_mun", "nom_ent", "pct_morena_2024", "pred", "resid", "tipologia_2024"]
top_mas = d.nlargest(12, "resid")[cols].round(1)
top_menos = d.nsmallest(12, "resid")[cols].round(1)
print("MÁS oficialista de lo esperado:"); display(top_mas)
print("MENOS oficialista de lo esperado:"); display(top_menos)

MÁS oficialista de lo esperado:


,id,nom_mun,nom_ent,pct_morena_2024,pred,resid,tipologia_2024
3373,121081,Cutzamala de Pinzón,Guerrero,96.4,37.6,58.8,bastion_oficialista
40237,071463,,Chiapas,100.0,42.9,57.1,bastion_oficialista
20350,202172,Santo Domingo Ingenio,Oaxaca,90.0,42.0,48.0,bastion_oficialista
27990,201955,Santiago Apoala,Oaxaca,71.8,24.0,47.9,bastion_oficialista
26048,070814,Nicolás Ruíz,Chiapas,99.7,54.3,45.4,bastion_oficialista
28111,202165,Santiago Zacatepec,Oaxaca,76.1,31.0,45.2,bastion_oficialista
21504,280347,Hidalgo,Tamaulipas,96.9,52.5,44.4,bastion_oficialista
13400,270656,Cunduacán,Tabasco,83.2,39.4,43.8,bastion_oficialista
24142,072102,San Fernando,Chiapas,57.4,14.7,42.7,bastion_oficialista
52503,155137,Toluca,México,75.1,33.7,41.4,bastion_oficialista


MENOS oficialista de lo esperado:


,id,nom_mun,nom_ent,pct_morena_2024,pred,resid,tipologia_2024
20427,072290,Francisco León,Chiapas,2.3,54.6,-52.3,dominante_moderada
42885,070054,Amatenango de la Frontera,Chiapas,3.1,54.9,-51.8,bastion_oficialista
16425,120078,Acapulco de Juárez,Guerrero,1.1,50.8,-49.6,competitiva
32799,070279,Comitán de Domínguez,Chiapas,6.6,55.5,-49.0,dominante_moderada
18649,201106,San Juan Cacahuatepec,Oaxaca,6.9,54.7,-47.8,dominante_moderada
9502,101429,Gómez Palacio,Durango,0.0,46.5,-46.5,bastion_oficialista
14409,161386,Pajacuarán,Michoacán de Ocampo,6.3,47.4,-41.2,dominante_moderada
18348,070882,Ocotepec,Chiapas,0.9,41.2,-40.4,dominante_moderada
24038,071927,Zinacantán,Chiapas,1.7,41.6,-39.9,bastion_oposicion
32122,082460,Ocampo,Chihuahua,11.5,51.1,-39.6,dominante_moderada


## 8. Zoom: un territorio con fuerte desviación

Ejemplo: las secciones sobre-performantes de un estado con residuo medio alto.

In [13]:
por_ent = d.groupby("nom_ent")["resid"].agg(["mean", "std", "count"]).sort_values("mean")
print("Residuo medio por entidad (pts):")
print(por_ent.round(2).to_string())

Residuo medio por entidad (pts):
                                  mean    std  count
nom_ent                                             
San Luis Potosí                 -13.63   6.80    910
Nuevo León                       -9.86   6.58   2714
Zacatecas                        -7.35   6.80    788
Jalisco                          -6.84   6.76   3209
Aguascalientes                   -6.33   4.89    609
Michoacán de Ocampo              -5.67   8.66   1733
Guanajuato                       -4.71   8.30   2306
Tlaxcala                         -4.22   7.01    467
Chiapas                          -4.00  11.60   1216
Campeche                         -3.74   9.80    356
Colima                           -3.62   5.88    322
Yucatán                          -3.47   8.47    986
Durango                          -3.02   6.75    755
Querétaro                        -1.75   7.41    796
Puebla                           -1.52   7.98   2194
Nayarit                          -1.50   6.82    614
Coahuila de Z

In [14]:
zoom_ent = por_ent.index[-1]
dz = d[d["nom_ent"] == zoom_ent].copy()
fig, ax = plt.subplots(figsize=(7, 6))
dz["geometry"] = dz.geometry.simplify(0.001)
dz.plot(column="resid", ax=ax, cmap="RdBu_r", legend=True, linewidth=0.1,
        vmin=-25, vmax=25, legend_kwds={"label": "residuo (pts)", "shrink": 0.7})
ax.set_title(f"Residuos SAR · {zoom_ent}\n(rojo = más oficialista que su perfil)", fontsize=11)
ax.set_axis_off()
fig.savefig(os.path.join(FIG, "sar07_zoom_entidad.png"), dpi=110, bbox_inches="tight")
plt.show()
plt.close(fig)

## 9. Exportar y conclusiones

In [15]:
out = d[["id", "ent", "mun", "pct_morena_2024", "pred", "resid",
           "categoria_sar", "lisa_resid", "tipologia_2024"]]
ruta = os.path.join(RAIZ, "data", "processed", "sar_residuales_seccion.csv")
out.to_csv(ruta, index=False)
print(f"guardado: {ruta}  ({len(out):,} secciones)")

guardado: /workspace/github__alglopez-jpg__ElctoGeoIA/data/processed/sar_residuales_seccion.csv  (52,976 secciones)


### Conclusiones

1. **El modelo funciona como radiografía**: ρ≈0.27 confirma dependencia espacial
   sustantiva; el perfil socioeconómico + entorno explica ~59% de la variación
   del voto a Morena por sección.
2. **La firma socioeconómica es clara**: falta de educación básica empuja el voto
   oficialista; automóvil e internet lo reducen.
3. **Los residuos son oro analítico**: ~±15 pts de desviación en miles de secciones
   revelan factores locales no capturados por el censo. Los clusters LISA los
   vuelven territorio accionable: zonas donde el oficialismo "sobre-rendimiento"
   sugiere máquinas territoriales; zonas de resistencia estructural sugieren
   identidades políticas locales.
4. **No es una bola de cristal**: el backtest (Fase 3) ya demostró que estos
   patrones territoriales no generalizan entre eras políticas. Su valor es
   explicar el *presente* y priorizar terreno, no predecir el futuro.

**Archivo generado**: `data/processed/sar_residuales_seccion.csv` — columna `resid`
(+ = más oficialista que su perfil) y clasificaciones `categoria_sar` / `lisa_resid`
para cruces con el panel maestro.